# small_fable — PLAN-FORMAT sweep: which FORMAT does a frozen model follow?

**Prior finding** (`GROUNDING_RESULTS.md`): a **frozen** Qwen2.5-1.5B-Instruct GROUNDS *concrete* English plans (conditional `neg_follow` ~59%) but **not** *generic / positional* ones (~10%), and merely *clearer* phrasing of the generic plan did not help. This notebook asks a sharper question.

**The experiment** (inference-only, frozen model, judge-free — we reuse the per-family solvers):
1. Hold the **same 100 problems** and the **same gold/neg answers** fixed.
2. Render each family's gold+neg plan in **5 FORMATS** — `terse`, `verbose`, `numbered`, `definitional` (these 4 are **generic**: positional/ordinal references only, **0 domain words**, hard-asserted via the `STRUCT_OK` gate) and `concrete` (**names the domain thing** — the established 59% upper bound).
3. Run the **same** `grounding_test.py` on each format file into its own out dir.
4. **Restrict to the rows the model fails UNAIDED** (`acc_noplan == 0`) — the rows where the plan is **load-bearing** (it *must* matter, because the model can't get the answer from the setup alone). `acc_noplan` depends only on (problem, model), not on the plan format, so this fails-unaided subset is **identical across formats**.
5. On that subset, read off the **TASK × FORMAT** following heatmap (`neg_follow` = followed the wrong plan to its wrong answer) and the per-task **3-way split** (follow / ignore→gold / botch→other) to see *which format each task reacts to* — and *why* some tasks ground and others don't.

> Reading the split: **ignore (→gold)** = ATTENTION failure (the override class — model can do the op, solves its own way) · **botch (→other)** = EXECUTION failure (the capability wall — model can't compute what the plan refers to).

## 0 · GPU check

In [ ]:
!nvidia-smi || echo 'NO GPU — runs on CPU (slow). Colab: Runtime -> GPU.'

## 1 · Clone & install (pulls the latest commit)

In [ ]:
import os
!rm -rf small_fable
!git clone -q https://github.com/sinha-k-prat/small_fable.git
%cd small_fable
!git log --oneline -1
!pip install -q -r requirements.txt
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('\nReady.')

## 2 · Generate all 5 format files (deterministic, instant)
One row-aligned JSONL per format. All 5 share identical `id`/`topic`/`problem`/`gold_answer`/`neg_answer` (same seed -> same RNG draws -> same 100 problems as `grounding_blocks_c1.jsonl`); they differ **only** in plan phrasing. The generator hard-asserts (a) cross-format alignment, (b) **0 domain leaks** for the 4 generic formats, and (c) `gold_answer != neg_answer`.

In [ ]:
!python -u tools/gen_plan_formats.py --n 100 --seed 7 --out_dir . --prefix grounding_fmt_

## 3 · One family's plan in each format, side by side
The **same** `constraint_select` problem rendered 5 ways. Note the perturbed block (block 1): the 4 generic formats reference the criterion **positionally** ("the 1st stated requirement"), while only `concrete` **names the domain thing** ("waterproof", "gadgets"). That naming is exactly the difference the sweep isolates.

In [ ]:
import json
FORMATS = ['terse', 'verbose', 'numbered', 'definitional', 'concrete']
GENERIC = ['terse', 'verbose', 'numbered', 'definitional']
rows = {f: [json.loads(l) for l in open(f'grounding_fmt_{f}.jsonl')] for f in FORMATS}
print(f'rows/format: ' + '  '.join(f'{f}={len(rows[f])}' for f in FORMATS))

# same problem across formats — show the first constraint_select row
rid = next(r['id'] for r in rows['verbose'] if r['topic'] == 'constraint_select')
base = next(r for r in rows['verbose'] if r['id'] == rid)
print(f'\n=== problem (id {rid}, topic {base["topic"]}) ===\n{base["problem"]}')
print(f'\ngold_answer = {base["gold_answer"]!r}   neg_answer = {base["neg_answer"]!r}')

for f in FORMATS:
    r = next(x for x in rows[f] if x['id'] == rid)
    tag = 'GENERIC — 0 domain words' if f in GENERIC else 'CONCRETE — NAMES the domain thing (upper bound)'
    print(f'\n---------- {f.upper()}  [{tag}] ----------')
    for i, blk in enumerate(r['gold_plan'], 1):
        print(f'  {i}. {blk}')

## 4 · Run the SAME probe on each format (`grounding_test.py`, unchanged)
One `--out` dir per format. A100 -> `bf16`; T4 -> `fp16`. Watch `marker_rate >= 90%` for each (the fraction of decodes that reached `FINAL ANSWER:`); if any is low (verbose/numbered/definitional are longer -> longer CoT), bump `--max_new_tokens` to 384 and rerun that one. Each run writes a `results.json` carrying a per-row `records` list that `analyze_formats.py` reads.

In [ ]:
DTYPE = 'fp16'  # 'bf16' on A100
for f in FORMATS:
    print(f'\n================ format = {f} ================\n')
    !python -u grounding_test.py --data grounding_fmt_{f}.jsonl --base Qwen/Qwen2.5-1.5B-Instruct \
        --dtype $DTYPE --device cuda --max_new_tokens 320 --out out_fmt_{f}

## 5 · Cross-format analysis (`tools/analyze_formats.py`)
Restricts to the **fails-unaided** subset (`acc_noplan == 0`, identical across formats), then prints the **TASK × FORMAT** following table (`neg_follow`), picks the best **generic** format (`concrete` excluded as the reference upper bound), prints its per-task **3-way split**, and saves the heatmap to `assets/plan_format_following.png`.

In [ ]:
!python tools/analyze_formats.py \
    --runs terse=out_fmt_terse verbose=out_fmt_verbose numbered=out_fmt_numbered \
           definitional=out_fmt_definitional concrete=out_fmt_concrete \
    --out assets/plan_format_following.png

## 6 · The TASK × FORMAT following heatmap
Rows = the 8 tasks, columns = the 5 plan formats, cell = `neg_follow` on the fails-unaided rows (brighter = the model **followed** the plan). The `concrete` column (right of the white divider) is the upper-bound reference. **Read off which format each task reacts to** — and which tasks stay dark in *every* generic format (the ones that don't ground).

In [ ]:
from IPython.display import Image, display
import os
p = 'assets/plan_format_following.png'
display(Image(filename=p)) if os.path.exists(p) else print('missing', p)

## 7 · Read-off — why some tasks ground and others don't

The TASK × FORMAT table + the per-task 3-way split (printed in §5) decompose every non-follow into one of two causes:

- **Follows in some/all generic formats** -> the primitive grounds **zero-shot**; format mostly modulates *how strongly*. (Established: `multi_hop_lookup` — a labelled-slot LOOKUP.)
- **Ignore (→gold) dominates** -> **ATTENTION failure** (the *override* class): the model **can** do the op, so it solves the problem its own way and weights the plan too little. A *clearer/definitional* format may help here, but the fix is attention training (instruction-following / contrastive SFT). (Established: `set_ops`, `categorize_rule`, `conditional_reco`, `scheduling`, `comparison_order`.)
- **Botch (→other) dominates** -> **EXECUTION failure** (the *capability wall*): the model **can't** compute what the plan refers to (e.g. build a total order from scattered facts), so **no format and no attention training fixes it** — it needs a bigger executor or a scratchpad. (Established: `transitive_logic`, `constraint_select`.)

The `concrete` column is the ceiling: where a generic format approaches it, the abstraction is viable zero-shot; where even `concrete` is low, the wall is **capability**, not phrasing. So the actionable conclusion: pick, per task, the lightest format that grounds — leave the easy primitives generic, reserve attention-training for the override class, and give the capability-class tasks a stronger executor.